In [58]:
import pandas as pd


# Colon Cancer Data Preparation

This notebook loads, validates, cleans, and reshapes the source datasets used by `Visualize.ipynb`. Run the cells from top to bottom to regenerate the dashboard-ready CSV files.

In [59]:
# Load the source datasets.
df = pd.read_csv("cases.csv")
df2 = pd.read_csv("Tobacco_Use.csv")

## Source data checks

Review dataset dimensions, column names, and missing values before cleaning.

In [60]:
# Compare the source dataset sizes.
dataset_summary = pd.DataFrame(
    {
        "Dataset": ["Cancer cases", "Tobacco use"],
        "Rows": [len(df), len(df2)],
        "Columns": [df.shape[1], df2.shape[1]],
    }
)
display(dataset_summary)

,Dataset,Rows,Columns
0,Cancer cases,74214,23
1,Tobacco use,1206,12


In [61]:
# Review the cancer dataset schema.
display(pd.DataFrame({"Column": df.columns}))

,Column
0,REGISTRY
1,SEX
2,CANCER
3,TOTAL
4,N0_4
5,N5_9
6,N10_14
7,N15_19
8,N20_24
9,N25_29


In [62]:
# Review the tobacco dataset schema.
display(pd.DataFrame({"Column": df2.columns}))

,Column
0,Indicator Name
1,Cancer Type
2,Year
3,Country Name
4,Region
5,Gender
6,Area
7,Numeric
8,Lower Confidence Limit
9,Upper Confidence Limit


In [63]:
# Count missing values in the cancer dataset.
cancer_missing = (
    df.isna()
    .sum()
    .rename("Missing values")
    .rename_axis("Column")
    .reset_index()
)
display(cancer_missing)

,Column,Missing values
0,REGISTRY,0
1,SEX,0
2,CANCER,0
3,TOTAL,0
4,N0_4,0
5,N5_9,0
6,N10_14,0
7,N15_19,0
8,N20_24,0
9,N25_29,0


In [64]:
# Count missing values in the tobacco dataset.
tobacco_missing = (
    df2.isna()
    .sum()
    .rename("Missing values")
    .rename_axis("Column")
    .reset_index()
)
display(tobacco_missing)

,Column,Missing values
0,Indicator Name,0
1,Cancer Type,1206
2,Year,0
3,Country Name,6
4,Region,0
5,Gender,0
6,Area,1206
7,Numeric,0
8,Lower Confidence Limit,0
9,Upper Confidence Limit,0


## Clean and validate the analysis datasets

Apply the filters and type conversions used by the visualizations, then check the resulting records.

In [65]:
# Remove unused fields and rows without a country identifier.
unused_tobacco_columns = ["Cancer Type", "Area", "Comment", "Unnamed: 11"]
df2_filtered = df2.drop(columns=unused_tobacco_columns).dropna(subset=["Country Name"]).copy()

print(f"Rows retained after removing missing country names: {len(df2_filtered):,}")
display(df2_filtered.head())

Rows retained after removing missing country names: 1,200


,Indicator Name,Year,Country Name,Region,Gender,Numeric,Lower Confidence Limit,Upper Confidence Limit
3,"Current tobacco use, adults aged 15+",2022,Afghanistan,Eastern Mediterranean Region,Total,22.7,16.2,29.3
4,"Current tobacco use, adults aged 15+",2022,Afghanistan,Eastern Mediterranean Region,Males,38.7,28,49.3
5,"Current tobacco use, adults aged 15+",2022,Afghanistan,Eastern Mediterranean Region,Females,6.8,4.4,9.2
6,"Current tobacco use, adults aged 15+",2022,African Region,African Region,Total,9.5,7.7,11.3
7,"Current tobacco use, adults aged 15+",2022,African Region,African Region,Males,16.6,13.6,19.6


In [66]:
# Keep records for colon cancer (code 13).
df_filtered = df.loc[df["CANCER"].eq(13)].copy()

print(f"Colon cancer records: {len(df_filtered):,} of {len(df):,}")
display(df_filtered.head())

Colon cancer records: 1,178 of 74,214


,REGISTRY,SEX,CANCER,TOTAL,N0_4,N5_9,N10_14,N15_19,N20_24,N25_29,...,N45_49,N50_54,N55_59,N60_64,N65_69,N70_74,N75_79,N80_84,N85,N_unk
12,101200399,1,13,109,0,0,0,0,0,0,...,7,8,11,9,22,13,11,5,5,0
75,101200399,2,13,122,0,0,1,0,1,2,...,18,18,15,19,8,6,9,3,7,0
138,101200799,1,13,221,0,0,0,0,1,2,...,13,18,11,32,36,31,26,17,12,0
201,101200799,2,13,196,0,0,0,0,0,2,...,15,19,26,25,28,13,16,13,10,0
264,120400199,1,13,29,0,0,0,0,0,0,...,5,3,4,3,3,5,2,0,0,0


### Colon cancer data checks

Validate the filtered age-group case counts before reshaping them for the dashboard.

In [67]:
# Check that cancer case counts are non-negative.
cancer_age_columns = [column for column in df_filtered.columns if column.startswith("N")]
negative_case_counts = df_filtered[cancer_age_columns].lt(0).sum().sum()

print(f"Negative age-group counts: {negative_case_counts:,}")
if negative_case_counts:
    display(df_filtered.loc[df_filtered[cancer_age_columns].lt(0).any(axis=1)].head())

Negative age-group counts: 0


In [68]:
# Confirm that case-count columns use integer data types.
cancer_count_types = pd.DataFrame(
    {
        "Data type": df_filtered[cancer_age_columns].dtypes.astype(str),
        "Is integer": [
            pd.api.types.is_integer_dtype(df_filtered[column])
            for column in cancer_age_columns
        ],
    }
)
display(cancer_count_types)
print(f"All case-count columns are integers: {cancer_count_types['Is integer'].all()}")

,Data type,Is integer
N0_4,int64,True
N5_9,int64,True
N10_14,int64,True
N15_19,int64,True
N20_24,int64,True
N25_29,int64,True
N30_34,int64,True
N35_39,int64,True
N40_44,int64,True
N45_49,int64,True


All case-count columns are integers: True


In [69]:
# map the registry number to an actual Country/city
registry_mapping = {
    101200399: "Algeria, Batna",
    101200799: "Algeria, Tizi Ouzou",
    120400199: "Benin, Cotonou",
    125009099: "France, La Réunion",
    140400199: "Kenya, Eldoret",
    140400299: "Kenya, Nairobi",
    148000099: "Mauritius",
    150400199: "Morocco, Casablanca",
    169000099: "Seychelles",
    171000199: "South Africa, Eastern Cape",
    171600170: "Zimbabwe, Bulawayo: African",
    171600270: "Zimbabwe, Harare: African",
    180000299: "Uganda, Kyadondo County",
    180000399: "Uganda, Gulu",
    203200599: "Argentina, Entre Ríos Province",
    203200799: "Argentina, Mendoza",
    207600299: "Brazil, Goiânia",
    207600399: "Brazil, Barretos",
    207600699: "Brazil, Campinas",
    207601199: "Brazil, Aracaju",
    207601299: "Brazil, Curitiba",
    207601399: "Brazil, João Pessoa",
    207601599: "Brazil, Recife",
    207601799: "Brazil, Jaú",
    207601899: "Brazil, Belo Horizonte",
    215200199: "Chile, Valdivia",
    215200399: "Chile, Region of Antofagasta",
    217000199: "Colombia, Cali",
    217000299: "Colombia, Bucaramanga",
    217000399: "Colombia, Manizales",
    217000499: "Colombia, Pasto",
    218800099: "Costa Rica",
    221800199: "Ecuador, Quito",
    221800399: "Ecuador, Guayaquil",
    221800599: "Ecuador, Manabí",
    225009199: "France, Martinique",
    225009299: "France, Guadeloupe",
    260400299: "Peru, Lima",
    263000099: "USA, Puerto Rico",
    278000099: "Trinidad and Tobago",
    285800099: "Uruguay",
    312400199: "Canada, Alberta",
    312400299: "Canada, British Columbia",
    312400399: "Canada, Manitoba",
    312400599: "Canada, New Brunswick",
    312400799: "Canada, Prince Edward Island",
    312400899: "Canada, Newfoundland and Labrador",
    312400999: "Canada, Yukon",
    312401099: "Canada, Ontario",
    312401199: "Canada, Quebec",
    312401299: "Canada, Saskatchewan",
    384000010: "USA: White",
    384000030: "USA: Black",
    384000099: "USA",
    384000210: "USA, California, San Francisco Bay Area: White",
    384000229: "USA, California, San Francisco Bay Area: Asian and Pacific Islander",
    384000230: "USA, California, San Francisco Bay Area: Black",
    384000250: "USA, California, San Francisco Bay Area: Hispanic White",
    384000260: "USA, California, San Francisco Bay Area: Non-Hispanic White",
    384000299: "USA, California, San Francisco Bay Area",
    384000310: "USA, California, Los Angeles County: White",
    384000320: "USA, California, Los Angeles County: Chinese",
    384000322: "USA, California, Los Angeles County: Korean",
    384000323: "USA, California, Los Angeles County: Filipino",
    384000329: "USA, California, Los Angeles County: Asian and Pacific Islander",
    384000330: "USA, California, Los Angeles County: Black",
    384000350: "USA, California, Los Angeles County: Hispanic White",
    384000360: "USA, California, Los Angeles County: Non-Hispanic White",
    384000399: "USA, California, Los Angeles County",
    384000410: "USA, Connecticut: White",
    384000430: "USA, Connecticut: Black",
    384000499: "USA, Connecticut",
    384000510: "USA, Georgia, Atlanta: White",
    384000530: "USA, Georgia, Atlanta: Black",
    384000599: "USA, Georgia, Atlanta",
    384000699: "USA, Iowa",
    384000710: "USA, Louisiana, New Orleans: White",
    384000730: "USA, Louisiana, New Orleans: Black",
    384000799: "USA, Louisiana, New Orleans",
    384000810: "USA, Michigan, Detroit: White",
    384000830: "USA, Michigan, Detroit: Black",
    384000899: "USA, Michigan, Detroit",
    384000910: "USA, New Mexico: White",
    384000942: "USA, New Mexico: American Indian",
    384000950: "USA, New Mexico: Hispanic White",
    384000960: "USA, New Mexico: Non-Hispanic White",
    384000999: "USA, New Mexico",
    384001299: "USA, Utah",
    384001399: "USA, Washington, Seattle",
    384001510: "USA, New Jersey: White",
    384001530: "USA, New Jersey: Black",
    384001599: "USA, New Jersey",
    384001610: "USA, New York State: White",
    384001629: "USA, New York State: Asian and Pacific Islander",
    384001630: "USA, New York State: Black",
    384001699: "USA, New York State",
    384001999: "USA, Kentucky",
    384002010: "USA, SEER (9 registries): White",
    384002030: "USA, SEER (9 registries): Black",
    384002099: "USA, SEER (9 registries)",
    384002210: "USA, Louisiana: White",
    384002230: "USA, Louisiana: Black",
    384002299: "USA, Louisiana",
    384002410: "USA, SEER (18 registries): White",
    384002429: "USA, SEER (18 registries): Asian and Pacific Islander",
    384002430: "USA, SEER (18 registries): Black",
    384002450: "USA, SEER (18 registries): Hispanic White",
    384002460: "USA, SEER (18 registries): Non-Hispanic White",
    384002499: "USA, SEER (18 registries)",
    384002510: "USA, California: White",
    384002529: "USA, California: Asian and Pacific Islander",
    384002530: "USA, California: Black",
    384002599: "USA, California",
    384004010: "USA, Arkansas: White",
    384004030: "USA, Arkansas: Black",
    384004099: "USA, Arkansas",
    384004210: "USA, Maryland: White",
    384004229: "USA, Maryland: Asian and Pacific Islander",
    384004230: "USA, Maryland: Black",
    384004299: "USA, Maryland",
    384004310: "USA, Minnesota: White",
    384004329: "USA, Minnesota: Asian and Pacific Islander",
    384004330: "USA, Minnesota: Black",
    384004399: "USA, Minnesota",
    384004499: "USA, Mississippi",
    384004510: "USA, North Carolina: White",
    384004529: "USA, North Carolina: Asian and Pacific Islander",
    384004530: "USA, North Carolina: Black",
    384004542: "USA, North Carolina: American Indian",
    384004599: "USA, North Carolina",
    384004699: "USA, North Dakota",
    384004710: "USA, Nebraska: White",
    384004730: "USA, Nebraska: Black",
    384004799: "USA, Nebraska",
    384004910: "USA, Nevada: White",
    384004929: "USA, Nevada: Asian and Pacific Islander",
    384004930: "USA, Nevada: Black",
    384004999: "USA, Nevada",
    384005043: "USA, Alaska: Alaska Natives",
    384005099: "USA, Alaska",
    384005110: "USA, Alabama: White",
    384005130: "USA, Alabama: Black",
    384005199: "USA, Alabama",
    384005210: "USA, Arizona: White",
    384005229: "USA, Arizona: Asian and Pacific Islander",
    384005230: "USA, Arizona: Black",
    384005242: "USA, Arizona: American Indian",
    384005299: "USA, Arizona",
    384005310: "USA, Colorado: White",
    384005329: "USA, Colorado: Asian and Pacific Islander",
    384005330: "USA, Colorado: Black",
    384005399: "USA, Colorado",
    384005410: "USA, District of Columbia: White",
    384005430: "USA, District of Columbia: Black",
    384005499: "USA, District of Columbia",
    384005510: "USA, Florida: White",
    384005529: "USA, Florida: Asian and Pacific Islander",
    384005530: "USA, Florida: Black",
    384005599: "USA, Florida",
    384005610: "USA, Georgia: White",
    384005629: "USA, Georgia: Asian and Pacific Islander",
    384005630: "USA, Georgia: Black",
    384005699: "USA, Georgia",
    384005799: "USA, Idaho",
    384005910: "USA, Indiana: White",
    384005930: "USA, Indiana: Black",
    384005999: "USA, Indiana",
    384006010: "USA, Massachusetts: White",
    384006029: "USA, Massachusetts: Asian and Pacific Islander",
    384006030: "USA, Massachusetts: Black",
    384006099: "USA, Massachusetts",
    384006199: "USA, Maine",
    384006210: "USA, Michigan: White",
    384006229: "USA, Michigan: Asian and Pacific Islander",
    384006230: "USA, Michigan: Black",
    384006299: "USA, Michigan",
    384006310: "USA, Missouri: White",
    384006330: "USA, Missouri: Black",
    384006399: "USA, Missouri",
    384006442: "USA, Montana: American Indian",
    384006499: "USA, Montana",
    384006510: "USA, Ohio: White",
    384006529: "USA, Ohio: Asian and Pacific Islander",
    384006530: "USA, Ohio: Black",
    384006599: "USA, Ohio",
    384006610: "USA, Oklahoma: White",
    384006630: "USA, Oklahoma: Black",
    384006642: "USA, Oklahoma: American Indian",
    384006699: "USA, Oklahoma",
    384006710: "USA, Oregon: White",
    384006729: "USA, Oregon: Asian and Pacific Islander",
    384006730: "USA, Oregon: Black",
    384006799: "USA, Oregon",
    384006910: "USA, Rhode Island: White",
    384006930: "USA, Rhode Island: Black",
    384006999: "USA, Rhode Island",
    384007010: "USA, South Carolina: White",
    384007030: "USA, South Carolina: Black",
    384007099: "USA, South Carolina",
    384007110: "USA, Texas: White",
    384007129: "USA, Texas: Asian and Pacific Islander",
    384007130: "USA, Texas: Black",
    384007199: "USA, Texas",
    384007299: "USA, Vermont",
    384007399: "USA, Washington State",
    384007499: "USA, West Virginia",
    384007699: "USA, Wyoming",
    384007899: "USA, South Dakota",
    384007910: "USA, Tennessee: White",
    384007930: "USA, Tennessee: Black",
    384007999: "USA, Tennessee",
    384008010: "USA, NPCR: White",
    384008029: "USA, NPCR: Asian and Pacific Islander",
    384008030: "USA, NPCR: Black",
    384008042: "USA, NPCR: American Indian",
    384008099: "USA, NPCR",
    404800046: "Bahrain: Bahrainis",
    409600099: "Brunei Darussalam",
    415600199: "China, Beijing City",
    415600299: "China, Qidong City",
    415600399: "China, Shanghai City",
    415600599: "China, Feicheng City",
    415600699: "China, Cixian County",
    415600799: "China, Jiashan County",
    415600899: "China, Linzhou City",
    415600999: "China, Wuhan City",
    415601099: "China, Baoding City",
    415601299: "China, Anshan City",
    415601499: "China, Zhongshan City",
    415601599: "China, Nangang District, Harbin City",
    415601699: "China, Shexian County",
    415601899: "China, Yanting County",
    415601999: "China, Yangzhong City",
    415602099: "China, Haining City",
    415602199: "China, Hangzhou City",
    415602299: "China, Dalian City",
    415602399: "China, Nanhu District, Jiaxing City",
    415602499: "China, Guangzhou City",
    415602599: "China, Haimen District, Nantong City",
    415603599: "China, Linqu County",
    415603799: "China, Cixi City",
    415603899: "China, Dafeng District, Yancheng City",
    415604199: "China, Dancheng County",
    415604499: "China, Dehui City",
    415604599: "China, Donghai County",
    415604799: "China, Ganyu District, Lianyungang City",
    415604899: "China, Ganzhou District, Zhangye City",
    415605099: "China, Gong'an County",
    415605199: "China, Guanyun County",
    415605299: "China, Guilin City",
    415605399: "China, Hefei City",
    415605499: "China, Hengdong County",
    415605599: "China, Hepu County",
    415605799: "China, Huaiyin District, Huai'an City",
    415606099: "China, Jiangmen City",
    415606299: "China, Jingtai County",
    415606399: "China, Jingxian County",
    415606599: "China, Jiyuan City",
    415606799: "China, Lianshui County",
    415606899: "China, Lianyungang City",
    415607199: "China, Liuzhou City",
    415607399: "China, Luoyang City",
    415607599: "China, Ma'anshan City",
    415607699: "China, Macheng City",
    415607899: "China, Neixiang County",
    415608099: "China, Chengdu City",
    415608199: "China, Shangyu District, Shaoxing City",
    415608499: "China, Sheyang County",
    415608699: "China, Tengzhou City",
    415609099: "China, Wu'an City",
    415609299: "China, Wuxi City",
    415609399: "China, Xianju County",
    415609499: "China, Xining City",
    415609699: "China, Xiping County",
    415609799: "China, Yangquan City",
    415609899: "China, Yanji City",
    415609999: "China, Yanshi City",
    415610199: "China, Yingdong District, Fuyang",
    415610199: 'China, Yingdong District, Fuyang City',
    415610299: 'China, Yueyanglou District, Yueyang City',
    415610399: 'China, Zhaoyuan City',
    415610499: 'China, Zhuanghe City',
    415610599: 'China, Zhuhai City',
    415611099: 'China, Anfu County',
    415611199: 'China, Anguo City',
    415611299: 'China, Aohan Banner, Chifeng City',
    415611399: 'China, Arongqi',
    415611699: 'China, Binhai County',
    415611799: 'China, Cangzhou City',
    415611899: 'China, Changfeng County',
    415612399: 'China, Changzhou City',
    415612699: 'China, Dangtu County',
    415612799: 'China, Danyang City',
    415613099: 'China, Dawukou District, Shizuishan City',
    415613199: 'China, Donggang County',
    415613299: 'China, Dongguan City',
    415613399: 'China, Dongtai City',
    415613499: 'China, Duanzhou District, Zhaoqing City',
    415613599: 'China, Evenki Autonomous Banner',
    415613699: 'China, Faku County',
    415613799: 'China, Fangcheng County',
    415613999: 'China, Feixi County',
    415614299: 'China, Fuqing City',
    415614499: 'China, Gaocheng City',
    415614599: 'China, Gaomi City',
    415614799: 'China, Guangan District, Guangan City',
    415614899: 'China, Guanghan City',
    415615099: 'China, Haian City',
    415615199: 'China, Hailar District, Hulun Buir City',
    415615599: 'China, Hebi City',
    415615699: 'China, Hengfeng County',
    415615799: 'China, Honghu City',
    415615999: 'China, Hongshan District, Chifeng City',
    415616099: 'China, Hongta District, Yuxi City',
    415616499: 'China, Yinzhou District, Ningbo City',
    415616599: 'China, Jiange County',
    415616699: 'China, Jiangyin City',
    415616799: 'China, Jiayu County',
    415616899: 'China, Jinan City',
    415617199: 'China, Jintan District, Changzhou City',
    415617299: 'China, Jiulongpo District, Chongqing City',
    415617499: 'China, Kaihua County',
    415617799: 'China, Kunshan City',
    415617899: 'China, Langzhong City',
    415618099: 'China, Liangshan County',
    415618499: 'China, Linhe District, Bayannur City',
    415618799: 'China, Liyang City',
    415618899: 'China, Longquan City',
    415619199: 'China, Lucheng District, Wenzhou City',
    415619399: 'China, Luoding City',
    415619499: 'China, Luohe City',
    415619599: 'China, Luquan City',
    415619699: 'China, Meihekou City',
    415619899: 'China, Mengjin County',
    415620099: 'China, Nanhai District, Foshan City',
    415620199: 'China, Nantong City',
    415620299: 'China, Nanxiong City',
    415620599: 'China, Qianxi County',
    415620799: 'China, Qingzhou City',
    415621199: 'China, Rudong County',
    415621299: 'China, Rugao City',
    415621399: 'China, Rural areas of Shanghai City',
    415621699: 'China, Ruyang County',
    415621799: 'China, Shan County',
    415622199: 'China, Shapingba District, Chongqing City',
    415622299: 'China, Shijiazhuang City',
    415622699: 'China, Shunde District, Foshan City',
    415622799: 'China, Song County',
    415622999: 'China, Suzhou City',
    415623599: 'China, Wangdu County',
    415623699: 'China, Wanzai County',
    415623799: 'China, Wuhu City',
    415623999: 'China, Wuzhou City',
    415624299: 'China, Xiangfu District, Kaifeng City',
    415624399: 'China, Xinan County',
    415624599: 'China, Xinji City',
    415624699: 'China, Xinluo District, Longyan City',
    415624799: 'China, Xinzhou District, Shangrao City',
    415624899: 'China, Xishan District, Kunming City',
    415625199: 'China, Xuyi County',
    415625299: 'China, Yaan City',
    415625399: 'China, Yakeshi City',
    415625499: 'China, Yancheng City',
    415625699: 'China, Yian District, Tongling City',
    415625899: 'China, Yiyuan County',
    415625999: 'China, Yongan City',
    415626099: 'China, Yongding District, Longyan City',
    415626199: 'China, Yongkang City',
    415626499: 'China, Yucheng County',
    415626699: 'China, Yunmeng County',
    415626799: 'China, Yunyang District, Shiyan City',
    415626899: 'China, Yuzhong District, Chongqing City',
    415627099: 'China, Zanhuang County',
    415627399: 'China, Zhangjiagang City',
    415627599: 'China, Zhengding County',
    415627699: 'China, Zhongxiang City',
    435600299: 'India, Bangalore',
    435600399: 'India, Mumbai',
    435600499: 'India, Chennai',
    435600699: 'India, Poona',
    435600799: 'India, Barshi, Paranda, and Bhum',
    435600899: 'India, Trivandrum',
    435601099: 'India, New Delhi',
    435601199: 'India, Dindigul, Ambilikkai',
    435601399: 'India, Aurangabad',
    435601499: 'India, Bhopal',
    435601599: 'India, Kamrup Urban District',
    435601699: 'India, Dibrugarh',
    435601799: 'India, Manipur',
    435601999: 'India, Mizoram',
    435602099: 'India, Wardha',
    435602199: 'India, Kollam',
    435602299: 'India, Tripura',
    435602499: 'India, Meghalaya',
    435603099: 'India, West Arunachal',
    435603499: 'India, Chandigarh',
    435603699: 'India, SAS Nagar',
    435603799: 'India, Sangrur District',
    435604099: 'India, Tamil Nadu',
    435604199: 'India, Ahmedabad, Urban',
    436400199: 'Iran (Islamic Republic of), Ardabil Province',
    436400399: 'Iran (Islamic Republic of), Golestan Province',
    437600090: 'Israel: Jews',
    437600094: 'Israel: Arabs',
    437600099: 'Israel',
    439200099: 'Japan',
    439200199: 'Japan, Hiroshima Prefecture',
    439200299: 'Japan, Miyagi Prefecture',
    439200499: 'Japan, Osaka Prefecture',
    439201099: 'Japan, Aichi Prefecture',
    439201499: 'Japan, Aomori Prefecture',
    439201599: 'Japan, Gunma Prefecture',
    439201699: 'Japan, Akita Prefecture',
    441000099: 'Republic of Korea',
    441000299: 'Republic of Korea, Busan',
    441000399: 'Republic of Korea, Seoul',
    441000499: 'Republic of Korea, Daegu',
    441000599: 'Republic of Korea, Daejeon',
    441000899: 'Republic of Korea, Ulsan',
    441000999: 'Republic of Korea, Jeju',
    441001099: 'Republic of Korea, Gwangju',
    441001199: 'Republic of Korea, Incheon',
    441400025: 'Kuwait: Kuwaitis',
    441400026: 'Kuwait: Non-Kuwaitis',
    441400099: 'Kuwait',
    460800299: 'Philippines, Manila',
    463400068: 'Qatar: Qatari',
    470200020: 'Singapore: Chinese',
    470200027: 'Singapore: Malay',
    470200040: 'Singapore: Indian',
    470200099: 'Singapore',
    476400199: 'Thailand, Chiang Mai',
    476400299: 'Thailand, Khon Kaen',
    476400399: 'Thailand, Bangkok',
    476400499: 'Thailand, Songkhla',
    476400599: 'Thailand, Lampang',
    476400799: 'Thailand, Lopburi Province',
    479200199: 'Turkey, Izmir',
    479200299: 'Turkey, Antalya',
    479200399: 'Turkey, Samsun',
    479200499: 'Turkey, Bursa',
    479200599: 'Turkey, Erzurum',
    479200699: 'Turkey, Eskişehir',
    479200799: 'Turkey, Edirne',
    479200899: 'Turkey, Trabzon',
    479201099: 'Turkey, Gaziantep',
    479201199: 'Turkey, Malatya',
    504000099: 'Austria',
    504000299: 'Austria, Carinthia',
    504000799: 'Austria, Tyrol',
    504000899: 'Austria, Vorarlberg',
    505600099: 'Belgium',
    511200099: 'Belarus',
    519100099: 'Croatia',
    519600099: 'Cyprus',
    520300099: 'Czech Republic',
    520800099: 'Denmark',
    523300099: 'Estonia',
    524600099: 'Finland',
    525000199: 'France, Bas-Rhin',
    525000299: 'France, Calvados',
    525000399: 'France, Doubs',
    525000499: 'France, Haut-Rhin',
    525000599: 'France, Isère',
    525000699: 'France, Somme',
    525000799: 'France, Tarn',
    525000899: 'France, Hérault',
    525000999: 'France, Limousin',
    525001099: 'France, Gironde',
    525001199: 'France, Territoire de Belfort',
    525001499: 'France, Loire-Atlantique',
    525001599: 'France, Poitou-Charentes',
    525001699: 'France, Manche',
    525001799: 'France, Vendée',
    525001899: 'France, Lille-Métropole',
    527600299: 'Germany, Hamburg',
    527600399: 'Germany, Bavaria',
    527600499: 'Germany, Baden-Württemberg',
    527600699: 'Germany, Lower Saxony',
    527600999: 'Germany, Rhineland-Palatinate',
    527601199: 'Germany, North Rhine-Westphalia',
    527601299: 'Germany, Bremen',
    527601699: 'Germany, Schleswig-Holstein',
    527603099: 'Germany, Saarland',
    535200099: 'Iceland',
    537200099: 'Ireland',
    538000199: 'Italy, Tuscany',
    538000499: 'Italy, Varese',
    538000699: 'Italy, Ragusa and Caltanissetta',
    538000899: 'Italy, Turin',
    538001399: 'Italy, Veneto',
    538001499: 'Italy, Umbria',
    538001599: 'Italy, Sassari',
    538001699: 'Italy, Naples South',
    538001899: 'Italy, Friuli-Venezia Giulia',
    538001999: 'Italy, Trento',
    538002299: 'Italy, Genova',
    538002599: 'Italy, Brescia',
    538002699: 'Italy, Milan',
    538002799: 'Italy, Salerno',
    538002899: 'Italy, Sondrio',
    538002999: 'Italy, Syracuse',
    538003199: 'Italy, Nuoro',
    538003399: 'Italy, Palermo',
    538003499: 'Italy, South Tyrol',
    538003799: 'Italy, Catania, Messina, and Enna',
    538004099: 'Italy, Lombardy, South, Pavia',
    538004299: 'Italy, Basilicata',
    538004399: 'Italy, Bergamo',
    538004499: 'Italy, Caserta',
    538004799: 'Italy, Brianza (Lecco and Monza e Brianza)',
    538005099: 'Italy, Aosta Valley',
    538005199: 'Italy, Avellino',
    538005299: 'Italy, Benevento',
    538005399: 'Italy, Calabria',
    538005599: 'Italy, Marche',
    538005799: 'Italy, Puglia',
    538005899: 'Italy, Molise',
    538005999: 'Italy, Naples Centre',
    538006099: 'Italy, Naples North',
    538006199: 'Italy, Mantova and Cremona',
    538006899: 'Italy, Emilia-Romagna',
    542800099: 'Latvia',
    543800099: 'Liechtenstein',
    544000099: 'Lithuania',
    547000099: 'Malta',
    552800099: 'The Netherlands',
    557800099: 'Norway',
    561600799: 'Poland, Kielce',
    562000599: 'Portugal, Azores',
    564300399: "Russian Federation, Karelia",
    564300499: "Russian Federation, Samara",
    564300699: "Russian Federation, Arkhangelsk",
    564300799: "Russian Federation, Pskov",
    564300999: "Russian Federation, Kaliningrad",
    564301099: "Russian Federation, Komi Republic",
    564301199: "Russian Federation, Murmansk",
    564301299: "Russian Federation, Vologda Region",
    564301399: "Russian Federation, Orenburg",
    570500099: "Slovenia",
    572400199: "Spain, Tarragona",
    572400299: "Spain, Granada",
    572400399: "Spain, Murcia",
    572400499: "Spain, Navarra",
    572400599: "Spain, Asturias",
    572400699: "Spain, Basque Country",
    572401099: "Spain, Girona",
    572401199: "Spain, Canary Islands",
    572401499: "Spain, La Rioja",
    572401599: "Spain, Castellón",
    572401799: "Spain, Salamanca",
    575200099: "Sweden",
    575600199: "Switzerland, Basel",
    575600299: "Switzerland, Geneva",
    575600499: "Switzerland, East",
    575600599: "Switzerland, Vaud",
    575600899: "Switzerland, Valais",
    575601099: "Switzerland, Ticino",
    575601199: "Switzerland, Graubünden and Glarus",
    575601299: "Switzerland, Fribourg",
    575601499: "Switzerland, Aargau",
    575601599: "Switzerland, Berne Solothurn",
    575601699: "Switzerland, Lucerne",
    575601799: "Switzerland, Neuchâtel and Jura",
    575601899: "Switzerland, Zurich and Zug",
    580400099: "Ukraine",
    582600199: "UK, England",
    582602099: "UK, Scotland",
    582603099: "UK, Northern Ireland",
    582604099: "UK, Wales",
    603600299: "Australia, NSW/ACT",
    603600399: "Australia, Queensland",
    603600499: "Australia, South Australia",
    603600599: "Australia, Tasmania",
    603600699: "Australia, Victoria",
    603600799: "Australia, Western Australia",
    603600862: "Australia, Northern Territory: Indigenous",
    603600863: "Australia, Northern Territory: Non-Indigenous",
    603600899: "Australia, Northern Territory",
    625009399: "France, New Caledonia",
    655400028: "New Zealand: Maori",
    655400029: "New Zealand: Pacific peoples",
    655400098: "New Zealand: Other",
    655400099: "New Zealand",
    684009910: "USA, Hawaii: White",
    684009920: "USA, Hawaii: Chinese",
    684009921: "USA, Hawaii: Japanese",
    684009923: "USA, Hawaii: Filipino",
    684009945: "USA, Hawaii: Hawaiian",
    684009999: "USA, Hawaii"
}

df_filtered['Location'] = df_filtered['REGISTRY'].map(registry_mapping)

# drop the registry column and put the Location column in front
df_filtered = df_filtered[['Location'] + [col for col in df.columns if col != 'Location']]
df_filtered = df_filtered.drop('REGISTRY', axis=1)
df_filtered.head()


,Location,SEX,CANCER,TOTAL,N0_4,N5_9,N10_14,N15_19,N20_24,N25_29,...,N45_49,N50_54,N55_59,N60_64,N65_69,N70_74,N75_79,N80_84,N85,N_unk
12,"Algeria, Batna",1,13,109,0,0,0,0,0,0,...,7,8,11,9,22,13,11,5,5,0
75,"Algeria, Batna",2,13,122,0,0,1,0,1,2,...,18,18,15,19,8,6,9,3,7,0
138,"Algeria, Tizi Ouzou",1,13,221,0,0,0,0,1,2,...,13,18,11,32,36,31,26,17,12,0
201,"Algeria, Tizi Ouzou",2,13,196,0,0,0,0,0,2,...,15,19,26,25,28,13,16,13,10,0
264,"Benin, Cotonou",1,13,29,0,0,0,0,0,0,...,5,3,4,3,3,5,2,0,0,0


In [70]:
# Check for duplicate colon cancer records.
cancer_duplicate_count = int(df_filtered.duplicated().sum())
print(f"Duplicate cancer records: {cancer_duplicate_count:,}")
if cancer_duplicate_count:
    display(df_filtered.loc[df_filtered.duplicated(keep=False)].head())

Duplicate cancer records: 0


### Tobacco data checks

Check the cleaned estimates and confidence limits used in the tobacco comparison.

In [71]:
# Convert tobacco estimates and confidence limits to numeric values.
tobacco_numeric_columns = [
    "Numeric",
    "Lower Confidence Limit",
    "Upper Confidence Limit",
]
for column in tobacco_numeric_columns:
    df2_filtered[column] = pd.to_numeric(df2_filtered[column], errors="coerce")

rows_before_numeric_cleanup = len(df2_filtered)
df2_filtered = df2_filtered.dropna(subset=tobacco_numeric_columns).copy()
rows_removed_numeric_cleanup = rows_before_numeric_cleanup - len(df2_filtered)

# Confirm that valid estimates are non-negative.
negative_tobacco_values = df2_filtered[tobacco_numeric_columns].lt(0).sum().sum()
print(f"Rows removed for invalid numeric values: {rows_removed_numeric_cleanup:,}")
print(f"Negative tobacco estimates or limits: {negative_tobacco_values:,}")
if negative_tobacco_values:
    display(df2_filtered.loc[df2_filtered[tobacco_numeric_columns].lt(0).any(axis=1)].head())

Rows removed for invalid numeric values: 192
Negative tobacco estimates or limits: 0


In [72]:
# Confirm that estimates and confidence limits have numeric data types.
tobacco_type_check = pd.DataFrame(
    {
        "Data type": df2_filtered[tobacco_numeric_columns].dtypes.astype(str),
        "Is numeric": [
            pd.api.types.is_numeric_dtype(df2_filtered[column])
            for column in tobacco_numeric_columns
        ],
    }
)
display(tobacco_type_check)
print(f"All estimate columns are numeric: {tobacco_type_check['Is numeric'].all()}")

,Data type,Is numeric
Numeric,float64,True
Lower Confidence Limit,float64,True
Upper Confidence Limit,float64,True


All estimate columns are numeric: True


In [73]:
# Check for duplicate tobacco records.
tobacco_duplicate_count = int(df2_filtered.duplicated().sum())
print(f"Duplicate tobacco records: {tobacco_duplicate_count:,}")
if tobacco_duplicate_count:
    display(df2_filtered.loc[df2_filtered.duplicated(keep=False)].head())

Duplicate tobacco records: 0


## Prepare data for the final visualizations

The remaining cells reshape the cleaned records and export the CSV files consumed by `Visualize.ipynb`.

### Optional reference export

Save the filtered location-level table before aggregating it for the dashboard.

In [74]:
# Save the filtered location-level incidence table for reference.
df_filtered.to_csv("colon_cancer_incidence.csv", index=False)

### Tobacco comparison data

Aggregate cancer counts by country and sex, join them to cleaned tobacco estimates, and reshape the result for the scatter plot.

In [75]:
# Aggregate location-level colon cancer counts to country and sex.
df_filtered["Country"] = df_filtered["Location"].str.split(",", n=1).str[0].str.strip()

grouped_data_colon = (
    df_filtered.groupby(["Country", "SEX"], as_index=False)
    .sum(numeric_only=True)
    .drop(columns="CANCER")
)
grouped_data_colon.head()

,Country,SEX,TOTAL,N0_4,N5_9,N10_14,N15_19,N20_24,N25_29,N30_34,...,N45_49,N50_54,N55_59,N60_64,N65_69,N70_74,N75_79,N80_84,N85,N_unk
0,Algeria,1,330,0,0,0,0,1,2,9,...,20,26,22,41,58,44,37,22,17,0
1,Algeria,2,318,0,0,1,0,1,4,9,...,33,37,41,44,36,19,25,16,17,0
2,Argentina,1,1692,0,0,0,3,4,11,22,...,54,100,149,253,241,280,213,176,106,0
3,Argentina,2,1543,0,1,0,4,4,20,14,...,78,102,139,180,213,174,205,185,164,0
4,Australia,1,26408,0,8,35,72,105,159,263,...,787,1289,1810,2790,3721,4233,4098,3335,2903,2


In [76]:
# Join cancer incidence with cleaned tobacco estimates.
df2_filtered = df2_filtered.rename(columns={"Country Name": "Country"})

merged_data = pd.merge(
    grouped_data_colon,
    df2_filtered,
    on="Country",
)

# Reshape cancer counts so each row represents a country, gender, and age group.
cancer_age_columns = [column for column in grouped_data_colon.columns if column.startswith("N")]
long_data = merged_data.melt(
    id_vars=["Country", "Gender", "Numeric"],
    value_vars=cancer_age_columns,
    var_name="Age_Group",
    value_name="Cancer_Incidence",
)

long_data.to_csv("long_data.csv", index=False)
print(f"Saved {len(long_data):,} rows to long_data.csv")

Saved 12,768 rows to long_data.csv


### Continent heatmap data

Map the joined country records to continents, aggregate by continent and gender, then reshape age groups for the heatmap.

In [77]:
# mapping country to continent
country_to_continent = {
    # Africa
    "Algeria": "Africa", "Angola": "Africa", "Benin": "Africa", "Botswana": "Africa",
    "Burkina Faso": "Africa", "Burundi": "Africa", "Cameroon": "Africa", "Cape Verde": "Africa",
    "Central African Republic": "Africa", "Chad": "Africa", "Comoros": "Africa", "Congo": "Africa",
    "Djibouti": "Africa", "Egypt": "Africa", "Equatorial Guinea": "Africa", "Eritrea": "Africa",
    "Eswatini": "Africa", "Ethiopia": "Africa", "Gabon": "Africa", "Gambia": "Africa",
    "Ghana": "Africa", "Guinea": "Africa", "Guinea-Bissau": "Africa", "Ivory Coast": "Africa",
    "Kenya": "Africa", "Lesotho": "Africa", "Liberia": "Africa", "Libya": "Africa",
    "Madagascar": "Africa", "Malawi": "Africa", "Mali": "Africa", "Mauritania": "Africa",
    "Mauritius": "Africa", "Morocco": "Africa", "Mozambique": "Africa", "Namibia": "Africa",
    "Niger": "Africa", "Nigeria": "Africa", "Rwanda": "Africa", "Sao Tome and Principe": "Africa",
    "Senegal": "Africa", "Seychelles": "Africa", "Sierra Leone": "Africa", "Somalia": "Africa",
    "South Africa": "Africa", "South Sudan": "Africa", "Sudan": "Africa", "Tanzania": "Africa",
    "Togo": "Africa", "Tunisia": "Africa", "Uganda": "Africa", "Zambia": "Africa", "Zimbabwe": "Africa",

    # Asia
    "Afghanistan": "Asia", "Armenia": "Asia", "Azerbaijan": "Asia", "Bahrain": "Asia",
    "Bangladesh": "Asia", "Bhutan": "Asia", "Brunei": "Asia", "Cambodia": "Asia",
    "China": "Asia", "Cyprus": "Asia", "Georgia": "Asia", "India": "Asia",
    "Indonesia": "Asia", "Iran": "Asia", "Iraq": "Asia", "Israel": "Asia",
    "Japan": "Asia", "Jordan": "Asia", "Kazakhstan": "Asia", "Kuwait": "Asia",
    "Kyrgyzstan": "Asia", "Laos": "Asia", "Lebanon": "Asia", "Malaysia": "Asia",
    "Maldives": "Asia", "Mongolia": "Asia", "Myanmar": "Asia", "Nepal": "Asia",
    "North Korea": "Asia", "Oman": "Asia", "Pakistan": "Asia", "Palestine": "Asia",
    "Philippines": "Asia", "Qatar": "Asia", "Saudi Arabia": "Asia", "Singapore": "Asia",
    "South Korea": "Asia", "Sri Lanka": "Asia", "Syria": "Asia", "Taiwan": "Asia",
    "Tajikistan": "Asia", "Thailand": "Asia", "Timor-Leste": "Asia", "Turkey": "Asia",
    "Turkmenistan": "Asia", "United Arab Emirates": "Asia", "Uzbekistan": "Asia",
    "Vietnam": "Asia", "Yemen": "Asia",

    # Europe
    "Albania": "Europe", "Andorra": "Europe", "Austria": "Europe", "Belarus": "Europe",
    "Belgium": "Europe", "Bosnia and Herzegovina": "Europe", "Bulgaria": "Europe",
    "Croatia": "Europe", "Czech Republic": "Europe", "Denmark": "Europe", "Estonia": "Europe",
    "Finland": "Europe", "France": "Europe", "Germany": "Europe", "Greece": "Europe",
    "Hungary": "Europe", "Iceland": "Europe", "Ireland": "Europe", "Italy": "Europe",
    "Kosovo": "Europe", "Latvia": "Europe", "Lithuania": "Europe", "Luxembourg": "Europe",
    "Malta": "Europe", "Moldova": "Europe", "Monaco": "Europe", "Montenegro": "Europe",
    "Netherlands": "Europe", "North Macedonia": "Europe", "Norway": "Europe", "Poland": "Europe",
    "Portugal": "Europe", "Romania": "Europe", "Russia": "Europe", "San Marino": "Europe",
    "Serbia": "Europe", "Slovakia": "Europe", "Slovenia": "Europe", "Spain": "Europe",
    "Sweden": "Europe", "Switzerland": "Europe", "Ukraine": "Europe", "United Kingdom": "Europe",
    "Vatican City": "Europe",

    # North America
    "Antigua and Barbuda": "North America", "Bahamas": "North America", "Barbados": "North America",
    "Belize": "North America", "Canada": "North America", "Costa Rica": "North America",
    "Cuba": "North America", "Dominica": "North America", "Dominican Republic": "North America",
    "El Salvador": "North America", "Grenada": "North America", "Guatemala": "North America",
    "Haiti": "North America", "Honduras": "North America", "Jamaica": "North America",
    "Mexico": "North America", "Nicaragua": "North America", "Panama": "North America",
    "Saint Kitts and Nevis": "North America", "Saint Lucia": "North America", "Saint Vincent and the Grenadines": "North America",
    "Trinidad and Tobago": "North America", "United States": "North America",

    # South America
    "Argentina": "South America", "Bolivia": "South America", "Brazil": "South America",
    "Chile": "South America", "Colombia": "South America", "Ecuador": "South America",
    "Guyana": "South America", "Paraguay": "South America", "Peru": "South America",
    "Suriname": "South America", "Uruguay": "South America", "Venezuela": "South America",

    # Oceania
    "Australia": "Oceania", "Fiji": "Oceania", "Kiribati": "Oceania", "Marshall Islands": "Oceania",
    "Micronesia": "Oceania", "Nauru": "Oceania", "New Zealand": "Oceania", "Palau": "Oceania",
    "Papua New Guinea": "Oceania", "Samoa": "Oceania", "Solomon Islands": "Oceania",
    "Tonga": "Oceania", "Tuvalu": "Oceania", "Vanuatu": "Oceania",

    # Antarctica
    "Antarctica": "Antarctica"
}

In [78]:
# Assign countries to continents, then aggregate counts and tobacco use.
merged_data["Continent"] = merged_data["Country"].map(country_to_continent)

age_group_columns = [
    column
    for column in merged_data.columns
    if column.startswith("N") and column != "Numeric"
]
continent_summary = (
    merged_data.groupby(["Continent", "Gender"])[age_group_columns]
    .sum()
    .reset_index()
)
continent_tobacco_usage = (
    merged_data.groupby(["Continent", "Gender"])["Numeric"]
    .mean()
    .reset_index()
)
continent_data = pd.merge(
    continent_summary,
    continent_tobacco_usage,
    on=["Continent", "Gender"],
)

# Reshape the continent summary for the heatmap.
melted = continent_data.melt(
    id_vars=["Continent", "Gender", "Numeric"],
    value_vars=age_group_columns,
    var_name="Age_Group",
    value_name="Cancer_Incidence",
)
melted["Age_Group"] = (
    melted["Age_Group"]
    .str.replace("N", "", regex=False)
    .str.replace("_", "-", regex=False)
)

melted.to_csv("melted.csv", index=False)
print(f"Saved {len(melted):,} rows to melted.csv")

Saved 342 rows to melted.csv


### Remaining dashboard input files

Prepare location-level incidence and country-level values for the map visualizations.

In [79]:
# Build the remaining CSV inputs loaded by Visualize.ipynb.
# Use readable age-group labels in the location-level incidence table.
incidence_by_location = df_filtered.copy()
incidence_by_location = incidence_by_location.rename(
    columns={
        column: column[1:].replace("_", "-")
        for column in incidence_by_location.columns
        if column.startswith("N")
    }
)
incidence_by_location.to_csv("incidence_by_location.csv", index=False)

# Prepare country-level salt-intake values for the choropleth map.
salt_source = pd.read_csv("Salt_Intake2.csv")
salt_by_country = (
    salt_source.loc[
        salt_source["Indicator Name"].eq("Mean population salt intake, adults aged 25+"),
        ["Country Name", "Numeric"],
    ]
    .rename(columns={"Country Name": "Country", "Numeric": "Salt_Intake"})
    .copy()
)
salt_by_country["Salt_Intake"] = pd.to_numeric(salt_by_country["Salt_Intake"], errors="coerce")
salt_by_country = salt_by_country.dropna(subset=["Country", "Salt_Intake"])
salt_by_country.to_csv("salt_by_country.csv", index=False)

# Aggregate colon cancer counts to country level for the choropleth map.
colon_source = pd.read_csv("colon_cancer_incidence2.csv")
colon_source["Country"] = colon_source["Location"].str.split(",", n=1).str[0].str.strip()
colon_by_country = (
    colon_source.groupby("Country", as_index=False)["TOTAL"]
    .sum()
    .rename(columns={"TOTAL": "Colon_Cancer_Rate"})
)
colon_by_country = colon_by_country.loc[
    ~colon_by_country["Country"].isin(["United States: Black", "United States: White"])
]
colon_by_country.to_csv("colon_by_country.csv", index=False)

print(f"Saved {len(incidence_by_location):,} location-level incidence rows")
print(f"Saved {len(salt_by_country):,} country-level salt-intake rows")
print(f"Saved {len(colon_by_country):,} country-level cancer rows")

Saved 1,178 location-level incidence rows
Saved 600 country-level salt-intake rows
Saved 70 country-level cancer rows
